# Exploratory Data Analysis

In [ ]:
#Importing essential libraries
from openpyxl import load_workbook
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


## Loading PCOS Dataset 

In [ ]:
#Loading the excel file
pcos_wb = load_workbook("PCOS Dataset.xlsx")

#Selecting the active sheet 
pcos_ws = pcos_wb.active

#Taking all rows as a list of list/tuples 
pcos_data = list(pcos_ws.iter_rows(values_only=True))

#Seperating the header from the actual data rows
headers = pcos_data[0]
data_rows = pcos_data[1:]

#Creating a pandas dataframe
pcos_df = pd.DataFrame(data_rows, columns=headers)

#Priting the table
print(pcos_df.to_string())

#Reading data from the cells
#for row in pcos_ws.iter_rows(values_only=True):
#    print(row)

## Viewing List of Feature Names

In [ ]:
#Viewing all feature names

#Loading the excel file
df = pd.read_excel("PCOS Dataset.xlsx")

#Extracting the column names
feature_names = df.columns.tolist()

#Printing the feature names
print(f"Total features: {len(feature_names)}\n")
for feature in feature_names:
    print(f"'{feature}',")

## Creating a reduced copy of the PCOS dataframe with only the essential features

In [ ]:
#Reduced Dataset

essential_features = [
    'Menstrual_Cycle_Length_days',
    'Menstrual_Irregularity',
    'BMI',
    'Fasting_Glucose_mg_dL',
    'Fasting_Insulin_uIU_mL',
    'HOMA_IR',
    'LH_mIU_mL',
    'FSH_mIU_mL',
    'LH_FSH_Ratio',
    'Total_Testosterone_ng_dL',
    'Free_Testosterone_pg_mL',
    'Total_Cholesterol_mg_dL',
    'HDL_mg_dL',
    'LDL_mg_dL',
    'Triglycerides_mg_dL',
    'Dietary_Sugar_Intake',
    'Physical_Activity_Level',
    'PCOS_Diagnosis', 
    'Hirsutism_Score_FG',
    'Acne_Severity',
    'Alopecia',
    'Skin_Darkening_Acanthosis',
]

df_2 = df[essential_features].copy()

df_2.head()

### Viewing information about the features

In [ ]:
#Viewing the split between PCOS and non PCOS patients
PCOS_patient_count = (df_2['PCOS_Diagnosis'] == 1).sum()
nonPCOS_patient_count = (df_2['PCOS_Diagnosis'] == 0).sum()

#Calculating percentages of each quantity
PCOS_percentage = ( PCOS_patient_count / 468 ) * 100
nonPCOS_percentage = ( nonPCOS_patient_count / 468 ) * 100

print(f"Count of PCOS patients: {PCOS_patient_count}")
print(f"Count of Non PCOS patients: {nonPCOS_patient_count}")
print("---")
print(f"Percentage of PCOS patients: {PCOS_percentage:.2f}%")
print(f"Percentage of Non PCOS patients: {nonPCOS_percentage:.2f}%")

In [ ]:
#Forcing pandas to show all the columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_2.describe()

### Creating box plots of numerical features to check for outliers

In [ ]:
#Inline plotting
%matplotlib inline
sns.set_theme(style="whitegrid") #setting white grid style for box plots

#Only creating boxplots of continuous numeric features
cont_features = []
for col in df_2.select_dtypes(include=['number']).columns:
    if df_2[col].nunique() > 2: #excluding the binary features which only have 2 unique values
        cont_features.append(col)

num_features = len(cont_features)
print(num_features)

#Making 4 features display per row
features_per_row = 4
num_rows = (num_features + features_per_row - 1) // features_per_row
#Figure size configuration
fig, axes = plt.subplots(nrows=num_rows, ncols=features_per_row, figsize=(20, num_rows * 4))
axes = axes.flatten()

#Iteration through every continuous feature and creating the boxplot
for i, col in enumerate(cont_features):
    sns.boxplot(data=df_2, y=col, ax=axes[i], color="paleturquoise", width=0.5)
    axes[i].set_title(f"Distribution of {col}", fontweight='bold')
    axes[i].set_ylabel("")

#Hiding blank subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

#Adjusting spaces and display
plt.tight_layout()
plt.show()


In [ ]:
#Checking if their are any missing values
na_count = df_2.isnull().sum()
print(na_count)

### Assessing Negative values of Continuous Numeric Features

In [ ]:
#It is impossible for these levels to be below zero

#Count of negative values in column Fasting Insulin 
fasting_insulin_neg_count = (df_2['Fasting_Insulin_uIU_mL'] < 0).sum()
print(f"Number of negative values in Fasting Insulin: {fasting_insulin_neg_count}")

#Count of negative values in column HOMA
homa_neg_count = (df_2['HOMA_IR'] < 0).sum()
print(f"Number of negative values in HOMA: {homa_neg_count}")

#Count of negative values in column LH
lh_neg_count = (df_2['LH_mIU_mL'] < 0).sum()
print(f"Number of negative values in LH: {lh_neg_count}")

#Count of negative values in column LH to FSH Ratio
lh_fsh_ratio_neg_count = (df_2['LH_FSH_Ratio'] < 0).sum()
print(f"Number of negative values in LH to FSH Ratio: {lh_fsh_ratio_neg_count}")

#Count of negative values in column Free Testosterone 
free_tes_neg_count = (df_2['Free_Testosterone_pg_mL'] < 0).sum()
print(f"Number of negative values in ree Testosterone: {free_tes_neg_count}")

#Count of negative values in column Triglycerides 
triglycerides_neg_count = (df_2['Triglycerides_mg_dL'] < 0).sum()
print(f"Number of negative values in Triglycerides: {triglycerides_neg_count}")


### Median Imputation of Negative Values within Features: Fasting_Insulin_uIU_mL, HOMA_IR, LH_mIU_mL, Free_Testosterone_pg_mL, Triglycerides_mg_dL

In [ ]:
#Median Imputation
#First change negative values to NaN 

cols = [
    'Fasting_Insulin_uIU_mL',
    'HOMA_IR',
    'LH_mIU_mL',
    'Free_Testosterone_pg_mL',
    'Triglycerides_mg_dL',
]

#Converting negative values to nan
df_2[cols] = df_2[cols].mask(df_2[cols] < 0, np.nan)

#Checking to see if negative values were removed
print("Remaining negative values: ")
print((df_2[cols] < 0).sum())


In [ ]:
#Converting NaN values to median values
for col in ['Fasting_Insulin_uIU_mL', 'HOMA_IR', 'LH_mIU_mL', 'Free_Testosterone_pg_mL', 'Triglycerides_mg_dL',]:
    col_median = df_2[col].median()
    df_2[col] = df_2[col].fillna(col_median)

#Checking that the dataset still contains 468 rows
print(f"Data shape: {df_2.shape}")

In [ ]:
#Recalculating LH to FSH Ratio column and HOMA IR column
#For mathematical integrity

#LH/FSH Ratio Formula
df_2['LH_FSH_Ratio'] = df_2['LH_mIU_mL'] / df_2['FSH_mIU_mL']

#HOMA IR Formula
df_2['HOMA_IR'] = (df_2['Fasting_Insulin_uIU_mL'] * df_2['Fasting_Glucose_mg_dL']) / 405

In [ ]:
df_2.head()

In [ ]:
#Forcing pandas to show all the columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_2.describe()

In [ ]:
#Count of zero values in column Fasting_Insulin_uIU_mL
fast_insu_zero_count = (df_2['Fasting_Insulin_uIU_mL'] == 0).sum()
print(f"Number of zero values in Fasting_Insulin_uIU_mL: {fast_insu_zero_count}")

#Count of zero values in column HOMA_IR
homa_zero_count = (df_2['HOMA_IR'] == 0).sum()
print(f"Number of zero values in HOMA_IR: {homa_zero_count}")

#Count of zero values in Triglycerides_mg_dL
triglycerides_zero_count = (df_2['Triglycerides_mg_dL'] == 0).sum()
print(f"Number of zero values in Triglycerides: {triglycerides_zero_count}")

In [ ]:
#Replacing zero values with NaN
for col in ['Fasting_Insulin_uIU_mL', 'Triglycerides_mg_dL']:
    df_2[col] = df_2[col].replace(0.0, np.nan)
    col_median = df_2[col].median()
    df_2[col] = df_2[col].fillna(col_median)
    
#Recalculating HOMA IR again
#HOMA IR Formula
df_2['HOMA_IR'] = (df_2['Fasting_Insulin_uIU_mL'] * df_2['Fasting_Glucose_mg_dL']) / 405

#Checking to see if zero values were removed
print("Remaining zero values: ")
print((df_2[['Fasting_Insulin_uIU_mL', 'Triglycerides_mg_dL', 'HOMA_IR']] == 0).sum())


In [ ]:
#Forcing pandas to show all the columns and rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df_2.describe()

### Identifying Outliers

In [ ]:
#
cont_cols = [ 'Fasting_Glucose_mg_dL', 'Fasting_Insulin_uIU_mL', 'HOMA_IR', 'LH_mIU_mL', 'FSH_mIU_mL', 'LH_FSH_Ratio', 'Total_Testosterone_ng_dL', 'Free_Testosterone_pg_mL', 'Total_Cholesterol_mg_dL', 'HDL_mg_dL', 'LDL_mg_dL', 'Triglycerides_mg_dL', 'Hirsutism_Score_FG']

outlier_data = []

for col in cont_cols: 
    
    #Calculating the quartiles and IQR
    Q1 = df_2[col].quantile(0.25)
    Q3 = df_2[col].quantile(0.75)
    IQR = Q3 - Q1

    #Calculating the lower and upper bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    #Finding the rows that breach the bounds
    outliers = df_2[(df_2[col] < lower_bound) | (df_2[col] > upper_bound)]
    outlier_count = len(outliers)
    percentage = (outlier_count / len(df_2)) * 100

    outlier_data.append({
        'Feature': col,
        'Number of Outliers': outlier_count,
        'Percentage of Outliers': percentage,
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2)
    })

#Compiling the outlier data for each feature into a dataframe
df_outliers = pd.DataFrame(outlier_data)

#Sorting the dataframe by order of the number of outliers
df_outliers = df_outliers.sort_values(by='Number of Outliers', ascending=False)

#Viewing the dataframe
df_outliers

### Creating Histograms to assess the distributions of features

In [ ]:
#Inline plotting
%matplotlib inline
sns.set_theme(style="whitegrid") #setting white grid style for box plots

#Only creating boxplots of continuous numeric features
cont_features = []
for col in df_2.select_dtypes(include=['number']).columns:
    if df_2[col].nunique() > 2: #excluding the binary features which only have 2 unique values
        cont_features.append(col)

num_features = len(cont_features)
print(num_features)

#Making 4 features display per row
features_per_row = 4
num_rows = (num_features + features_per_row - 1) // features_per_row
#Figure size configuration
fig, axes = plt.subplots(nrows=num_rows, ncols=features_per_row, figsize=(20, num_rows * 4))
axes = axes.flatten()

#Iteration through every continuous feature and creating the histogram
for i, col in enumerate(cont_features):
    sns.histplot(data=df_2, x=col, ax=axes[i], kde=True, color="dodgerblue", bins='auto')
    axes[i].set_title(f"Distribution of {col}", fontsize=12, fontweight='bold')
    axes[i].set_xlabel("") #Getting rid of redundant x axis label
    axes[i].set_ylabel("Frequency", fontsize=10)

    #Hiding blank subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
#Saving the cleaned dataset to a new file
df_2.to_excel('clean_pcos_data.xlsx', index=False)